# Covariates in Difference-in-Differences: The LaLonde Test in Python

Reproducing **Scott Cunningham's** ["Covariates, diff in diff and LaLonde test"](https://causalinf.substack.com/p/covariates-diff-in-diff-and-lalonde) in Python.

We estimate the ATT of a job-training program **eight ways** on the LaLonde/Dehejia-Wahba non-experimental panel (185 NSW trainees + 15,992 CPS controls) and compare each to the experimental benchmark of **\$1,794**. The lesson: covariates rescue a DiD estimate only when they enter the control group's counterfactual **trend**.

Companion post: [Covariates in DiD — the LaLonde test](https://carlos-mendez.org/post/python_did_covariates_lalonde/) · Intro: [Introduction to DiD in Python](https://carlos-mendez.org/post/python_did/).

In [ ]:
# Colab setup — install the analysis stack (skip locally if already installed)
%pip install -q pyfixest causaldata diff-diff statsmodels

## Setup

`pyfixest` for the regressions, `causaldata` for the data, `diff-diff` for a package cross-check. Seed `90210` matches the reference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pyfixest as pf

RANDOM_SEED = 90210
BENCHMARK = 1794.0
XVARS = ["age", "agesq", "agecube", "educ", "educsq",
         "marr", "nodegree", "black", "hisp", "re74", "u74"]
XF = " + ".join(XVARS)
STEEL_BLUE, WARM_ORANGE, TEAL, GREY = "#6a9bcc", "#d97757", "#00a89e", "#9aa0a6" 

## Data: the LaLonde-DW non-experimental panel

Non-experimental sample = 185 NSW **treated** + 15,992 CPS controls. The 260 experimental controls are held out for the benchmark only. We reshape to a 2-period panel (post=0 → `re75`, post=1 → `re78`).

In [ ]:
from causaldata import nsw_mixtape, cps_mixtape

nsw = nsw_mixtape.load_pandas().data
cps = cps_mixtape.load_pandas().data
wide = pd.concat([nsw[nsw.treat == 1], cps], ignore_index=True)
wide["ever_treated"] = wide["treat"].astype(int)
for c in ["age","educ","marr","nodegree","black","hisp","re74","re75","re78"]:
    wide[c] = wide[c].astype(float)
wide["agesq"] = wide.age**2; wide["agecube"] = wide.age**3
wide["educsq"] = wide.educ**2; wide["u74"] = (wide.re74 == 0).astype(float)
wide["dy"] = wide.re78 - wide.re75
wide["id"] = np.arange(len(wide))

pre  = wide.assign(re=wide.re75, post=0.0)
post = wide.assign(re=wide.re78, post=1.0)
panel = pd.concat([pre, post], ignore_index=True)
pd.crosstab(panel.ever_treated, panel.post)

## The experimental benchmark

On the 445-row experimental sample, `re78 ~ treat` gives the canonical Dehejia-Wahba ATT of **\$1,794** — our ground truth.

In [ ]:
exp = nsw.copy()
for c in ["re74","re75","re78"]: exp[c] = exp[c].astype(float)
bench = pf.feols("re78 ~ treat", data=exp, vcov="HC1").tidy()
print("Experimental benchmark ATT = $%.0f (SE %.0f)" %
      (bench.loc["treat","Estimate"], bench.loc["treat","Std. Error"]))

## Helper: pull the DiD coefficient

In [ ]:
def att(fit):
    td = fit.tidy()
    key = [k for k in td.index if "post" in k.lower() and "ever_treated" in k.lower()][0]
    return td.loc[key, "Estimate"], td.loc[key, "Std. Error"]

## Spec 0 — Naive TWFE (no covariates) → the number to beat

In [ ]:
s0 = pf.feols("re ~ post * ever_treated", data=panel, vcov="HC1")
print("Spec 0 (naive)      ATT = $%.0f (SE %.0f)" % att(s0))

## Spec A — Additive X (covariates in the *level*)

Two-way fixed effects with controls. Time-invariant covariates are swept out by the within transformation, so this is **inert**.

In [ ]:
sA = pf.feols(f"re ~ post * ever_treated + {XF}", data=panel, vcov="HC1")
print("Spec A (additive X) ATT = $%.0f (SE %.0f)" % att(sA))

## Spec BT — X × treatment (covariates in the *effect*)

Heterogeneous treatment effects via g-computation. Saturating in **levels** does nothing for conditional parallel trends → still **inert**.

In [ ]:
d = panel.assign(T=panel.post * panel.ever_treated)
T_ints = " + ".join(f"T:{x}" for x in XVARS)
fBT = pf.feols(f"re ~ post + ever_treated + T + {XF} + {T_ints}", data=d, vcov="HC1")
tau = fBT.predict(newdata=d.assign(T=1.0)) - fBT.predict(newdata=d.assign(T=0.0))
attBT = tau[((d.ever_treated==1)&(d.post==1)).values].mean()
print("Spec BT (X x treatment) ATT = $%.0f" % attBT)

## Spec B — X × post (covariates in the *trend*) → the correction

Interacting covariates with `post` lets different workers be on different earnings trajectories. **The estimate collapses toward the benchmark.**

In [ ]:
post_ints = " + ".join(f"post:{x}" for x in XVARS)
sB = pf.feols(f"re ~ post * ever_treated + {XF} + {post_ints}", data=panel, vcov="HC1")
print("Spec B (X x post)   ATT = $%.0f (SE %.0f)" % att(sB))

## Spec C — Saturated first differences = HIT (1997)

First-difference, then saturate on `D × X`. A covariate's coefficient is now its effect on the **trend** — two corrections in one regression.

In [ ]:
D_ints = " + ".join(f"ever_treated:{x}" for x in XVARS)
sC = pf.feols(f"dy ~ {XF} + {D_ints} + ever_treated", data=wide, vcov="HC1")
tauC = (sC.predict(newdata=wide.assign(ever_treated=1.0))
        - sC.predict(newdata=wide.assign(ever_treated=0.0)))
attC = tauC[wide.ever_treated.values == 1].mean()
print("Spec C (saturated FD = HIT) ATT = $%.0f" % attC)

## HIT (1997) by hand — control-only outcome regression

Fit \$\Delta y\$ on the **controls only**, impute the counterfactual change for the treated, average the gains. Numerically identical to Spec C.

In [ ]:
mH = smf.ols(f"dy ~ {XF}", data=wide[wide.ever_treated == 0]).fit()
attHIT = (wide.dy - mH.predict(wide))[wide.ever_treated == 1].mean()
print("HIT by hand ATT = $%.0f" % attHIT)

## IPW — Abadie (2005)

Model **treatment** with a propensity score and reweight the controls:
\$w_i = \dfrac{D_i - \hat p(X_i)}{1 - \hat p(X_i)} \cdot \dfrac{1}{\Pr(D=1)}\$.

In [ ]:
Xc = sm.add_constant(wide[XVARS])
wide["phat"] = sm.Logit(wide.ever_treated, Xc).fit(disp=0).predict(Xc)
p = wide.ever_treated.mean()
w_ipw = (wide.ever_treated - wide.phat) / (1 - wide.phat) / p
attIPW = (w_ipw * wide.dy).mean()
print("IPW (Abadie 2005) ATT = $%.0f" % attIPW)

## DR — Sant'Anna-Zhao (2020)

Doubly robust: outcome regression **and** propensity weighting. Consistent if either model is right.

In [ ]:
wide["dyhat"] = mH.predict(wide)
dr_t = (wide.ever_treated * (wide.dy - wide.dyhat) / p).mean()
dr_c = ((1 - wide.ever_treated) * (wide.phat/(1-wide.phat)) * (wide.dy - wide.dyhat) / p).mean()
attDR = dr_t - dr_c
print("DR (Sant'Anna-Zhao 2020) ATT = $%.0f" % attDR)

## The payoff: the covariate arc

Assemble everything and plot it against the **\$1,794** benchmark.

In [ ]:
rows = [("No covariates (naive)", att(s0)[0], att(s0)[1], "inert"),
        ("Additive X (level)",    att(sA)[0], att(sA)[1], "inert"),
        ("X x treatment (effect)", attBT,     att(s0)[1], "inert"),
        ("X x post (trend)",       att(sB)[0], att(sB)[1], "corrected"),
        ("Saturated FD = HIT",     attC,       att(sB)[1], "corrected"),
        ("IPW (Abadie)",           attIPW,     att(sB)[1], "propensity"),
        ("Doubly robust",          attDR,      att(sB)[1], "propensity")]
res = pd.DataFrame(rows, columns=["estimator","att","se","group"])
res.round(0)

In [ ]:
colors = {"inert": GREY, "corrected": STEEL_BLUE, "propensity": TEAL}
markers = {"inert": "o", "corrected": "D", "propensity": "s"}
o = res.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.axvline(BENCHMARK, color=WARM_ORANGE, ls="--", lw=2, label="RCT benchmark $1,794")
for i, r in o.iterrows():
    ax.errorbar(r.att, i, xerr=1.96*r.se, fmt=markers[r.group], color=colors[r.group],
                ms=11, capsize=4, lw=2, mec="white")
ax.set_yticks(range(len(o))); ax.set_yticklabels(o.estimator)
ax.set_xlim(0, 4200)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f"${v:,.0f}"))
ax.set_xlabel("ATT estimate (95% CI)")
ax.set_title("Covariates rescue LaLonde only when they enter the trend", weight="bold")
ax.legend(loc="lower right", frameon=False)
for s in ["top","right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Cross-check with the diff-diff package

Independent validation with [`diff-diff`](https://github.com/igerber/diff-diff).

In [ ]:
from diff_diff import DifferenceInDifferences, CallawaySantAnna
dd = DifferenceInDifferences(cluster="id", seed=90210).fit(
    panel, outcome="re", treatment="ever_treated", time="post", unit="id")
cs_df = panel.assign(first_treat=np.where(panel.ever_treated == 1, 1, 0))
cs = CallawaySantAnna(estimation_method="dr", seed=90210).fit(
    cs_df, outcome="re", unit="id", time="post", first_treat="first_treat", covariates=XVARS)
print("diff-diff naive 2x2      = $%.0f" % dd.att)
print("diff-diff Callaway-SA DR = $%.0f  (by-hand DR $%.0f)" % (float(np.atleast_1d(cs.att)[0]), attDR))

## Takeaways

- **Where** a covariate enters, not **whether** it is included, decides everything: level and effect are inert (~\$3,621); the **trend** corrects (~\$1,711–1,993, near the \$1,794 truth).
- The ubiquitous **additive-controls TWFE** is exactly the specification that fails.
- Trust the **pattern**, not the decimal — the CIs are wide and the benchmark is itself noisy.

Source: Cunningham (2026), *Covariates, diff in diff and LaLonde test*, Scott's Mixtape Substack.